# Day I11 — 신호처리 II: 시간 정규화, ensemble average, 그리고 센서 융합

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I11.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I11.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 한 참가자의 왼발 VGRF를 stride로 잘라 ensemble 평균 내기

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")                              # 노트북에서는 이 두 줄을 빼도 된다
import matplotlib.pyplot as plt

fs = 100
d = np.loadtxt("data/gaitpdb/GaCo01_01.txt")
left = d[:, 17]                                    # 왼발 총 힘(N)

# 1) stride 나누기: 20 N 문턱을 넘는 순간 = IC (간단판. Day I12에서 규칙을 다듬는다)
on = left > 20
ic = np.flatnonzero(~on[:-1] & on[1:]) + 1
dur = np.diff(ic) / fs
med = np.median(dur)
keep = (dur > 0.8 * med) & (dur < 1.2 * med)       # 중앙값의 ±20% 밖 stride는 제외
keep[:2] = keep[-2:] = False                       # 처음·마지막 2 stride(가속·감속)도 제외
print(f"IC {len(ic)}개 → stride {len(dur)}개 → 사용 {keep.sum()}개 (중앙값 {med:.2f} s)")

# 2) stride마다 101점(0, 1, …, 100 %GC)으로 시간 정규화
pct = np.linspace(0, 100, 101)
curves = []
for a, b in zip(ic[:-1][keep], ic[1:][keep]):
    x = left[a:b + 1]                              # IC부터 다음 IC까지(양 끝 포함)
    curves.append(np.interp(pct, np.linspace(0, 100, len(x)), x))
curves = np.array(curves)                          # (stride 수, 101)
mean, sd = curves.mean(axis=0), curves.std(axis=0, ddof=1)
print("정규화 행렬 모양:", curves.shape)
lens = (np.diff(ic)[keep] + 1)
print(f"정규화 전 stride 길이(표본 수): {lens.min()}–{lens.max()}")

# 3) 평균 곡선이 숨기는 것
early = pct <= 30                                  # 첫 봉우리는 stance 앞부분에 있다
print(f"평균 곡선의 첫 봉우리: {mean[early].max():.0f} N ({pct[early][np.argmax(mean[early])]:.0f}%GC)")
print(f"각 stride 첫 봉우리의 평균: {curves[:, early].max(axis=1).mean():.0f} N")
to_pct = np.array([pct[np.argmax((c < 20) & (pct > 30))] for c in curves])   # 30%GC 뒤 처음 20 N 아래
print(f"stride별 toe-off 위치: 평균 {to_pct.mean():.1f}%GC, 범위 {to_pct.min():.0f}–{to_pct.max():.0f}%GC")
print(f"평균 곡선이 20 N 아래로 내려가는 위치: {pct[np.argmax((mean < 20) & (pct > 30))]:.0f}%GC")

plt.figure(figsize=(6, 3.5))
plt.plot(pct, mean, color="k", label="mean")
plt.fill_between(pct, mean - sd, mean + sd, alpha=0.3, label="±1 SD (between strides)")
plt.xlabel("% gait cycle (left IC → next left IC)"); plt.ylabel("left total force (N)")
plt.legend(); plt.tight_layout()
plt.savefig("I11_ensemble.png", dpi=120)

### 예시 2 — complementary filter로 자이로 drift 잡기 (가상)

In [ ]:
import numpy as np

rng = np.random.default_rng(11)
fs = 100
dt = 1 / fs
t = np.arange(0, 60, dt)
# 가상 정강이 기울기(°): 1 Hz로 ±30° 흔들리는 걷기 + 천천히 바뀌는 자세 성분
true = 30 * np.sin(2 * np.pi * 1.0 * t) + 5 * np.sin(2 * np.pi * 0.05 * t)
true_rate = np.gradient(true, dt)                       # 참 각속도(°/s)

gyro = true_rate + 0.5 + rng.normal(0, 1.0, len(t))     # bias 0.5 °/s + 잡음
# 가속도계 기울기: 평균적으로는 맞지만, 걸을 때 선가속도 때문에 크게 흔들린다(가상: 8° 잡음)
acc_angle = true + rng.normal(0, 8.0, len(t))

gyro_only = true[0] + np.cumsum(gyro) * dt               # 자이로만 적분

def complementary(gyro, acc_angle, tau, dt, start):
    alpha = tau / (tau + dt)                             # tau(초)보다 느린 변화는 가속도계를 믿는다
    est = np.empty_like(acc_angle)
    est[0] = start
    for k in range(1, len(est)):
        est[k] = alpha * (est[k - 1] + gyro[k] * dt) + (1 - alpha) * acc_angle[k]
    return est

def rms(e): return np.sqrt(np.mean(e ** 2))
last = t >= 50                                           # 마지막 10초에서 비교
print(f"자이로만 적분   : 마지막 10초 RMS 오차 {rms(gyro_only[last] - true[last]):5.1f}°")
print(f"가속도계 기울기 : 마지막 10초 RMS 오차 {rms(acc_angle[last] - true[last]):5.1f}°")
for tau in [0.1, 1.0, 10.0]:
    est = complementary(gyro, acc_angle, tau, dt, acc_angle[0])
    print(f"complementary τ = {tau:4.1f} s (α = {tau / (tau + dt):.4f}): RMS 오차 {rms(est[last] - true[last]):5.1f}°")